In [33]:
from math import exp, sin, log, copysign

Функция используемая в лабораторной работе

In [34]:
def func(x):
    return exp(sin(x) * log(x))

Реализация методов (Задание №1)

Все методы возвращают пару (x,y), соответствующие минимальному значению функции

In [35]:
def dichotomy_method(f: callable, a: float, b: float, epsilon=1e-6) -> (float, float):
    n = 0
    tol = epsilon/2
    intervals = []
    while abs(b - a) > epsilon:
        n += 1
        intervals.append((a, b))
        c = (a + b)/2
        if f(c - tol) < f(c + tol):
            b = c
        else:
            a = c
    print(f"intervals on every iteration: {intervals}")
    print(f"required number of iterations for epsilon = {epsilon}: {n}")
    return (a + b)/2, f((a + b)/2)


def golden_section_method(f: callable, a: float, b: float, epsilon=1e-6) -> (float, float):
    phi = (3 - 5**0.5)/2
    x1 = a + (b - a) * phi
    x2 = b - (b - a) * phi
    n = 0
    intervals = []
    f1, f2 = f(x1), f(x2)
    while abs(b - a) > epsilon:
        n += 1
        intervals.append((a, b))
        if f1 < f2:
            b = x2
            x2 = x1
            x1 = a + (b - a) * phi
            f1, f2 = f(x1), f1
        else:
            a = x1
            x1 = x2
            x2 = b - (b - a) * phi
            f1, f2 = f2, f(x2)
    print(f"intervals on every iteration: {intervals}")
    print(f"required number of iterations for epsilon = {epsilon}: {n}")
    return (a + b)/2, f((a + b)/2)


def fibonacci_method(f: callable, a: float, b: float, n: int) -> (float, float):
    fib = [1, 1]
    for i in range(2, n + 1):
        fib.append(fib[i - 1] + fib[i - 2])

    intervals = []

    k = 1
    x1 = a + (b - a) * fib[n - k - 1] / fib[n - k + 1]
    x2 = a + (b - a) * fib[n - k] / fib[n - k + 1]
    f1, f2 = f(x1), f(x2)

    while k < n - 1:
        intervals.append((a, b))
        if f1 < f2:
            b = x2
            x2 = x1
            f2 = f1
            k += 1
            x1 = a + (b - a) * fib[n - k - 1] / fib[n - k + 1]
            f1 = f(x1)
        else:
            a = x1
            x1 = x2
            f1 = f2
            k += 1
            x2 = a + (b - a) * fib[n - k] / fib[n - k + 1]
            f2 = f(x2)

    print(f"intervals on every iteration: {intervals}")
    if f1 < f2:
        return x1, f(x1)
    else:
        return x2, f(x2)


def parabolic_interpolation(f: callable, x1: float, x3: float, epsilon=1e-6) -> (float, float):
    x2 = (x1 + x3)/2
    f1, f2, f3 = f(x1), f(x2), f(x3)
    n = 0
    intervals = []
    while abs(x3 - x1) > epsilon:
        n += 1
        intervals.append((x1, x3))
        u = x2 - ((x2 - x1) ** 2 * (f2 - f3) - (x2 - x3) ** 2 * (f2 - f1)) \
            / (2 * ((x2 - x1) * (f2 - f3) - (x2 - x3) * (f2 - f1)))
        fu = f(u)
        if x2 <= u:
            if f2 <= fu:
                x1, x2, x3 = x1, x2, u
                f1, f2, f3 = f1, f2, fu
            else:
                x1, x2, x3 = x2, u, x3
                f1, f2, f3 = f2, fu, f3
        else:
            if fu <= f2:
                x1, x2, x3 = x1, u, x2
                f1, f2, f3 = f1, fu, f2
            else:
                x1, x2, x3 = u, x2, x3
                f1, f2, f3 = fu, f2, f3

    print(f"intervals on every iteration: {intervals}")
    print(f"required number of iterations for epsilon = {epsilon}: {n}")
    return (x1 + x3)/2, f((x1 + x3)/2)


def brent_method(f: callable, a: float, c: float, epsilon=1e-6) -> (float, float):
    phi = (3 - 5 ** 0.5)/2
    x = w = v = a + (c - a) * phi
    fx = fw = fv = f(x)
    d = e = c - a
    n = 0
    intervals = []
    while abs(d) > epsilon:
        n += 1
        intervals.append((a, c))
        g, e = e, d
        if x != w and x != v and w != v and fx != fw and fx != fv and fv != fw:
            # здесь параболическая аппроксимация
            if v < w:
                x1, x3 = v, w
                f1, f3 = fv, fw
            else:
                x1, x3 = w, v
                f1, f3 = fw, fv
            u = x - ((x - x1) ** 2 * (fx - f3) - (x - x3) ** 2 * (fx - f1)) \
                / (2 * ((x - x1) * (fx - f3) - (x - x3) * (fx - f1)))
            if a + epsilon < u < c - epsilon and abs(u - x) < g/2:
                true_u = u
                d = abs(true_u - x)
        else:
            if x < (c - a)/2:
                true_u = x + (c - x) * phi
                d = c - x
            else:
                true_u = x - (x - a) * phi
                d = x - a
        if abs(true_u - x) < epsilon:
            true_u = x + copysign(epsilon, true_u - x)
        fu = f(true_u)
        if fu <= fx:
            if true_u >= x:
                a = x
            else:
                c = x
            v, w, x = w, x, true_u
            fv, fw, fx = fw, fx, fu
        else:
            if true_u >= x:
                c = true_u
            else:
                a = true_u
            if fu <= fw or w == x:
                v, w = w, true_u
                fv, fw = fw, fu
            elif fu <= fv or v == x or v == w:
                v, fv = true_u, fu

    print(f"intervals on every iteration: {intervals}")
    print(f"required number of iterations for epsilon = {epsilon}: {n}")
    return x, f(x)

Задание №2

In [36]:
x1, x2 = 8, 20
accuracies = [1e-4]
for accuracy in accuracies:
    print(f"minimal value found by dichotomy_method: {dichotomy_method(func, x1, x2, accuracy)}\n")
    print(f"minimal value found by golden_section_method: {golden_section_method(func, x1, x2, accuracy)}\n")
    n = 0
    fib1 = 1
    fib2 = 1
    while True:
        n += 1
        a = (x2-x1)/accuracy
        if a < fib2:
            break
        tmp = fib1
        fib1 = fib2
        fib2 += tmp
    print(f"required number of iterations for epsilon {accuracy}: {n}")
    print(f"minimal value found by fibonacci_method: {fibonacci_method(func, x1, x2, n)}\n")
    print(f"minimal value found by parabolic_interpolation: {parabolic_interpolation(func, x1, x2, accuracy)}\n")
    print(f"minimal value found by brent_method: {brent_method(func, x1, x2, accuracy)}\n\n")

intervals on every iteration: [(8, 20), (8, 14.0), (11.0, 14.0), (11.0, 12.5), (11.0, 11.75), (11.0, 11.375), (11.0, 11.1875), (11.0, 11.09375), (11.0, 11.046875), (11.0234375, 11.046875), (11.0234375, 11.03515625), (11.029296875, 11.03515625), (11.0322265625, 11.03515625), (11.0322265625, 11.03369140625), (11.032958984375, 11.03369140625), (11.032958984375, 11.0333251953125), (11.03314208984375, 11.0333251953125)]
required number of iterations for epsilon = 0.0001: 17
minimal value found by dichotomy_method: (11.033279418945312, 0.0907896809404432)

intervals on every iteration: [(8, 20), (8, 15.416407864998739), (8, 12.583592135001261), (9.750776405003785, 12.583592135001261), (9.750776405003785, 11.50155281000757), (10.41951348501388, 11.50155281000757), (10.832815729997476, 11.50155281000757), (10.832815729997476, 11.246117974981072), (10.832815729997476, 11.088250565023975), (10.930383155066876, 11.088250565023975), (10.990683139954573, 11.088250565023975), (10.990683139954573, 11